In [1]:
import torch
import sys
sys.path.append("examples")

from toy_transformer import ToyTransformer
from modellens import ModelLens

model = ToyTransformer()
input_ids = torch.randint(0, 100, (1, 10))

lens = ModelLens(model)
print(lens)
print(f"Backend: {lens.adapter.type_of_adapter}")
print(f"\nLayers:")
for name in lens.layer_names():
    print(f"  {name}")

ModelLens(backend=pytorch, hooks=0, params=162,980)
Backend: pytorch

Layers:
  embed
  blocks
  blocks.0
  blocks.0.ln_1
  blocks.0.attn
  blocks.0.attn.out_proj
  blocks.0.ln_2
  blocks.0.mlp
  blocks.0.mlp.0
  blocks.0.mlp.1
  blocks.0.mlp.2
  blocks.1
  blocks.1.ln_1
  blocks.1.attn
  blocks.1.attn.out_proj
  blocks.1.ln_2
  blocks.1.mlp
  blocks.1.mlp.0
  blocks.1.mlp.1
  blocks.1.mlp.2
  blocks.2
  blocks.2.ln_1
  blocks.2.attn
  blocks.2.attn.out_proj
  blocks.2.ln_2
  blocks.2.mlp
  blocks.2.mlp.0
  blocks.2.mlp.1
  blocks.2.mlp.2
  ln_f
  lm_head


In [2]:
# 1. Hooks — basic activation capture
lens.attach_all()
output = lens.run(input_ids)
print(f"Output shape: {output.shape}")
print(f"Activations captured: {len(lens.get_activations())}")
print(f"Shapes:")
for name, shape in lens.hooks.get_shapes().items():
    print(f"  {name}: {shape}")

Output shape: torch.Size([1, 10, 100])
Activations captured: 27
Shapes:
  embed: torch.Size([1, 10, 64])
  blocks.0.ln_1: torch.Size([1, 10, 64])
  blocks.0.attn: torch.Size([1, 10, 64])
  blocks.0.ln_2: torch.Size([1, 10, 64])
  blocks.0.mlp.0: torch.Size([1, 10, 256])
  blocks.0.mlp.1: torch.Size([1, 10, 256])
  blocks.0.mlp.2: torch.Size([1, 10, 64])
  blocks.0.mlp: torch.Size([1, 10, 64])
  blocks.0: torch.Size([1, 10, 64])
  blocks.1.ln_1: torch.Size([1, 10, 64])
  blocks.1.attn: torch.Size([1, 10, 64])
  blocks.1.ln_2: torch.Size([1, 10, 64])
  blocks.1.mlp.0: torch.Size([1, 10, 256])
  blocks.1.mlp.1: torch.Size([1, 10, 256])
  blocks.1.mlp.2: torch.Size([1, 10, 64])
  blocks.1.mlp: torch.Size([1, 10, 64])
  blocks.1: torch.Size([1, 10, 64])
  blocks.2.ln_1: torch.Size([1, 10, 64])
  blocks.2.attn: torch.Size([1, 10, 64])
  blocks.2.ln_2: torch.Size([1, 10, 64])
  blocks.2.mlp.0: torch.Size([1, 10, 256])
  blocks.2.mlp.1: torch.Size([1, 10, 256])
  blocks.2.mlp.2: torch.Size([1,

In [3]:
from modellens.analysis.logit_lens import run_logit_lens, decode_logit_lens

# Create a simple vocab dict for decoding (0-99)
vocab = {i: str(i) for i in range(100)}
results = run_logit_lens(lens, input_ids, top_k=5)
decoded = decode_logit_lens(results, vocab=vocab)

for layer_name, predictions in list(decoded.items())[:5]:
    tokens_str = ", ".join(f"{tok} ({prob:.3f})" for tok, prob in predictions)
    print(f"{layer_name:30s} -> {tokens_str}")

embed                          -> 4 (0.034), 88 (0.031), 98 (0.031), 2 (0.028), 99 (0.022)
blocks.0.ln_1                  -> 4 (0.040), 88 (0.030), 98 (0.030), 2 (0.026), 99 (0.023)
blocks.0.attn                  -> 7 (0.012), 44 (0.012), 18 (0.011), 39 (0.011), 24 (0.011)
blocks.0.ln_2                  -> 4 (0.043), 88 (0.031), 98 (0.030), 2 (0.028), 17 (0.025)
blocks.0.mlp.2                 -> 77 (0.013), 12 (0.013), 91 (0.013), 39 (0.013), 8 (0.012)


In [4]:
# Residual stream
from modellens.analysis.residual_stream import run_residual_analysis, identify_critical_layers

block_layers = ["blocks.0", "blocks.1", "blocks.2"]
lens.attach_layers(block_layers)
residual_results = run_residual_analysis(lens, input_ids, layer_names=block_layers)

for name, data in residual_results["contributions"].items():
    print(f"{name:15s} | rel: {data['relative_contribution']:.3f} | cos: {data['cosine_similarity']:.3f}")

blocks.1        | rel: 0.238 | cos: 0.971
blocks.2        | rel: 0.229 | cos: 0.973


In [5]:
# Embeddings
from modellens.analysis.embeddings import run_embeddings_analysis

embed_results = run_embeddings_analysis(lens, input_ids)
print(f"\nEmbedding dim: {embed_results['embed_dim']}")
print(f"Sequence length: {embed_results['seq_length']}")
print(f"Similarity matrix shape: {embed_results['similarity_matrix'].shape}")


Embedding dim: 64
Sequence length: 10
Similarity matrix shape: torch.Size([10, 10])


In [6]:
from modellens.analysis.attention import run_attention_analysis, head_summary

attn_results = run_attention_analysis(lens, input_ids)
print(f"Number of attention layers: {attn_results['num_layers']}")

for name, data in attn_results["attention_maps"].items():
    print(f"{name}: heads={data['num_heads']}, seq={data['seq_length']}")

Number of attention layers: 3
blocks.0.attn: heads=10, seq=10
blocks.1.attn: heads=10, seq=10
blocks.2.attn: heads=10, seq=10


In [7]:
summary = head_summary(attn_results)
for name, data in list(summary.items())[:5]:
    print(f"\n{name}:")
    print(f"  Entropy:       {[f'{e:.2f}' for e in data['entropy']]}")
    print(f"  Max attention: {[f'{a:.2f}' for a in data['max_attention']]}")


blocks.0.attn:
  Entropy:       ['2.26', '2.29', '2.26', '2.28', '2.27', '2.28', '2.27', '2.22', '2.29', '2.29']
  Max attention: ['0.18', '0.14', '0.18', '0.14', '0.15', '0.14', '0.13', '0.22', '0.12', '0.13']

blocks.1.attn:
  Entropy:       ['2.28', '2.28', '2.28', '2.28', '2.25', '2.27', '2.26', '2.29', '2.25', '2.27']
  Max attention: ['0.15', '0.13', '0.14', '0.14', '0.16', '0.13', '0.16', '0.13', '0.15', '0.16']

blocks.2.attn:
  Entropy:       ['2.25', '2.29', '2.27', '2.27', '2.26', '2.28', '2.25', '2.29', '2.26', '2.25']
  Max attention: ['0.17', '0.12', '0.14', '0.14', '0.14', '0.14', '0.16', '0.12', '0.14', '0.17']
